# Delete (undeploy) a MAAP DPS algorithm

The MAAP **Register Algorithm** GUI can register but **not** delete. Undeploying is a
`DELETE` against the OGC processes API, which is what this notebook does.

**Search for it, then delete it by ID** — that's the whole notebook:

1. **Step 1** — run it to connect (needs a MAAP token).
2. **Step 2** — set `WHAT_ARE_YOU_SEARCHING_FOR` to any part of the algorithm name
   (`"blackmarble"`, `"umbra"`, `"ogc-test"`, ...) and run. You get a table of matches
   with their **`processID`**.
3. **Step 4** — put the `processID` ints you want gone in `DELETE_IDS`, run once with
   `DRY_RUN = True` to confirm the targets, then flip it to `False`.
4. **Step 5** — re-list to confirm they're gone.

Step 3 is optional (narrow further by who deployed it).

**When you need it**

- An algorithm was **renamed** (e.g. `umbra` / `umbra-ogc-test` ->
  `disasters-umbra-process`). A rename registers a **NEW** process — the old one is
  left behind and must be deleted by hand, or the Submit Jobs dropdown keeps showing
  stale entries that still run old code.
- Throwaway `-ogc-test` experiments.
- Failed / partial registrations.

**Rules of the road**

- You delete by **`processID`** (an int MAAP assigns at registration), **not** by
  `name:version`. Passing a name just `404`s — hence search first, delete second.
- One `processID` per **name+version**, so an algorithm registered at two versions has
  two IDs and needs two deletes.
- **Only the deployer can delete their own process** — anything else returns `403`.
- Deletion removes the *process registration* only. Job history, outputs already in
  `nasa-disasters-staging`, and the built container are untouched.
- Re-registering the same name afterwards gets a **new** `processID`.

**Prereqs**

- Runs on the MAAP hub / ADE (needs MAAP auth). Kernel: the **`disasters_dps`** conda
  env — the same env `dps/*/build-env.sh` builds, which pins `maap-py`. If that kernel
  isn't in the picker:
  ```bash
  conda run -n disasters_dps pip install ipykernel
  conda run -n disasters_dps python -m ipykernel install --user --name disasters_dps \
      --display-name "Python 3 (disasters_dps)"
  ```
- A valid MAAP token. `MAAP()` reads it from the workspace; if `token` comes back empty,
  set **Settings -> MAAP Settings -> `maapToken`** (your `MAAP_PGT`), or
  `export MAAP_PGT=...` before launching the kernel.

Full write-up: [`docs/DPS.md` -> Deleting (undeploying) an algorithm](../docs/DPS.md#deleting-undeploying-an-algorithm).

## 1. Connect

In [ ]:
import json
import requests
from maap.maap import MAAP

# OGC processes API.
# NOTE the /api suffix. This is NOT the same string as the JupyterLab extensions'
# `maapApiUrl` setting, which must have NO /api suffix (the extensions append it).
BASE = "https://api.maap-project.org/api/ogc/processes"

maap = MAAP()
headers = maap._get_api_header()   # private helper: carries `token` (+ MAAP_PGT proxy-ticket)

assert headers.get("token"), (
    "No MAAP token. Set Settings -> MAAP Settings -> maapToken (your MAAP_PGT), "
    "or export MAAP_PGT before starting the kernel."
)
print("auth header fields:", sorted(headers))

## 2. Search

Set **`WHAT_ARE_YOU_SEARCHING_FOR`** to any part of the algorithm name — `"blackmarble"`,
`"umbra"`, `"ogc-test"`, `"disasters"`. Case doesn't matter, and blank (`""`) lists every
registered process.

The **`processID`** column is what you paste into step 4. The rest of the table is there
to help you pick the right row: `name:version` (an algorithm registered at two versions
has two IDs — deleting one leaves the other), `deployedBy` (only the deployer can
delete), and when it was last modified.

Searching is read-only — re-run it as often as you like.

In [ ]:
WHAT_ARE_YOU_SEARCHING_FOR = "blackmarble"   # substring of the process id; "" = list everything


def fetch_processes():
    r = requests.get(BASE, headers=headers)
    r.raise_for_status()
    return r.json()["processes"]


def matching(rows, term=None):
    """Rows whose process id contains `term` (default: WHAT_ARE_YOU_SEARCHING_FOR). Blank = all."""
    term = (WHAT_ARE_YOU_SEARCHING_FOR if term is None else term).lower()
    return [p for p in rows if term in p["id"].lower()]


def show(rows):
    """Print processID / name:version / deployer / last modified, sorted by ID."""
    if not rows:
        print("(none)")
        return
    for p in sorted(rows, key=lambda x: int(x["processID"])):
        name = f'{p["id"]}:{p["version"]}'
        print(f'processID={str(p["processID"]):<5} {name:<42} '
              f'deployedBy={str(p["deployedBy"]):<16} {p.get("lastModifiedTime", "")}')


# procs stays the FULL listing on purpose -- step 4 looks up processIDs in it and step 5
# diffs against it. Only the printout is filtered.
procs = fetch_processes()
hits = matching(procs)
print(f"{len(procs)} processes registered, "
      f"{len(hits)} matching {WHAT_ARE_YOU_SEARCHING_FOR!r}\n")
show(hits)

In [ ]:
# Full shape of one entry, if you need a field the table above doesn't show.
# Prefers a search hit so the example matches what you're actually looking at.
example = (hits or procs)[0]
print("KEYS:", list(example.keys()), "\n")
print(json.dumps(example, indent=2, default=str))

## 3. Optional — narrow to what *you* deployed

Skip this if step 2 already showed you the `processID` you want.

Same `WHAT_ARE_YOU_SEARCHING_FOR` as step 2, plus `ME`: set it to your MAAP username (the
`deployedBy` value in the table above) to hide everyone else's. Blank = don't filter by
deployer. Useful on a busy account — you can only delete your own, so anything that isn't
yours would come back `403`.

In [ ]:
ME = ""   # e.g. "kdl0040" — blank = don't filter by deployer

mine = [p for p in matching(procs) if not ME or p["deployedBy"] == ME]
print(f"{len(mine)} of {len(procs)} match "
      f"(ME={ME!r}, WHAT_ARE_YOU_SEARCHING_FOR={WHAT_ARE_YOU_SEARCHING_FOR!r})\n")
show(mine)

## 4. Delete by processID

Copy the **`processID`** ints out of the table above into `DELETE_IDS` — e.g.
`DELETE_IDS = [48, 53]`. Ints, not names: the API deletes by ID only, and a name just
`404`s.

Run once with `DRY_RUN = True`. It echoes each target back as
`processID name:version (deployedBy=...)` so you can read what you are about to remove.
When that list looks right, set `DRY_RUN = False` and run again.

An ID that isn't in the current step-2 listing is **refused** rather than sent — the guard
against a stale ID left over from an earlier run. If you hit that, re-run step 2.

Response codes: **200** deleted · **403** you are not the deployer · **404** no such
`processID` (already gone, or you passed the name instead of the ID).

In [ ]:
DELETE_IDS = []     # e.g. [30, 31] — processID ints from the tables above
DRY_RUN = True      # flip to False to actually delete

by_id = {str(p["processID"]): p for p in procs}

for pid in DELETE_IDS:
    p = by_id.get(str(pid))
    if p is None:
        print(f"{pid} -> not in the current listing; refusing (re-run step 2)")
        continue
    label = f'{pid} {p["id"]}:{p["version"]} (deployedBy={p["deployedBy"]})'
    if DRY_RUN:
        print("DRY RUN would delete:", label)
        continue
    r = requests.delete(f"{BASE}/{pid}", headers=headers)
    print(label, "->", r.status_code, r.text)
    # 200 = undeployed | 403 = not the deployer | 404 = no such processID

## 5. Verify

Re-lists and prints `removed processIDs`, diffed against the snapshot taken in step 2 —
that line is the confirmation the delete actually took. The table under it is filtered by
`WHAT_ARE_YOU_SEARCHING_FOR` again, so you are looking at the same set you started from,
minus what you deleted.

It also refreshes the snapshot, so you can go straight back to step 4 with more IDs.

Last check: the MAAP **Submit Jobs -> Process** dropdown. That dropdown is the real
"is it deployed" answer.

In [ ]:
after = fetch_processes()
removed = {str(p["processID"]) for p in procs} - {str(p["processID"]) for p in after}

print(f"{len(procs)} -> {len(after)} processes")
print("removed processIDs:", sorted(removed) or "none")
print()
show(matching(after))   # printout filtered by WHAT_ARE_YOU_SEARCHING_FOR

procs = after   # refresh the FULL snapshot so step 4 can be re-run against current state
hits = matching(procs)

## Notes

- **The search matches the process id only** — not the version, not the deployer. Use
  `ME` in step 3 to filter by deployer.
- **Delete is per-`processID`, so a name with several versions has several IDs.** Deleting
  `foo:dev` leaves `foo:v1.2.0` alone.
- **A rename is not a move.** Editing `algorithm_name` in `algorithm_config.yaml` /
  `dps/ogc/<name>.yml` and re-registering creates a second process; the old name keeps
  running the code it was built with until you delete it here.
- **`403` on your own algorithm** usually means the token belongs to a different MAAP
  account than the one that registered it (e.g. registered from the ADE, deleting from
  the Disasters hub). Check `deployedBy`.
- **Deleting does not cancel running jobs** and does not remove outputs from
  `nasa-disasters-staging/dps_output/<activation_event>/`.
- *Editing this notebook:* the search filters the **printout**, never `procs`. Step 4
  resolves IDs against `procs` and step 5 diffs against it, so filtering the snapshot
  itself would make step 4 refuse valid targets and break the removed-IDs check.